In [1]:
import os
import numpy as np

# Force CPU usage to prevent the 'CudnnRNN' driver error in your Intel environment
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

from tensorflow.keras.preprocessing.text import text_to_word_sequence, Tokenizer
from tensorflow.keras.utils import pad_sequences, to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

def train_sonnet_network(file_path='sonnet.txt', epochs=500):
    """
    Builds, trains, and returns a sequential LSTM network for text generation.
    """
    #Load and preprocess text
    with open(file_path, 'r') as file:
        text = file.read()
        lines = text.lower().split('\n')

    #Tokenize text
    words = text_to_word_sequence(text)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(words)
    vocabulary_size = len(tokenizer.word_index) + 1

    #Create n-gram subsequences
    sequences = tokenizer.texts_to_sequences(lines)
    subsequences = []
    for sequence in sequences:
        for i in range(1, len(sequence)):
            subsequence = sequence[:i+1]
            subsequences.append(subsequence)

    #Pad sequences
    max_sequence_length = max([len(seq) for seq in subsequences])
    padded_sequences = pad_sequences(subsequences, maxlen=max_sequence_length, padding='pre')

    #Split into features (x) and labels (y)
    x = padded_sequences[:, :-1]
    y = padded_sequences[:, -1]
    y = to_categorical(y, num_classes=vocabulary_size)

    #Build the network architecture
    model = Sequential()
    
    model.add(Embedding(input_dim=vocabulary_size, output_dim=100, input_length=max_sequence_length - 1))
    model.add(LSTM(150, return_sequences=False))
    model.add(Dense(vocabulary_size, activation='softmax'))

    #Compile and fit the model
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(x, y, epochs=epochs, verbose=1)

    # Return the trained model object along with tokenizer metrics if needed later
    return model


sonnet_model = train_sonnet_network('sonnet.txt', epochs=500)


C:\Users\ander\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.0238 - loss: 6.8598
Epoch 2/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 12s 25ms/step - accuracy: 0.0302 - loss: 6.4470
Epoch 3/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 12s 25ms/step - accuracy: 0.0402 - loss: 6.2598
Epoch 4/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.0465 - loss: 6.0614
Epoch 5/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.0532 - loss: 5.8508
Epoch 6/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.0635 - loss: 5.6164 
Epoch 7/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.0748 - loss: 5.3685
Epoch 8/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.0874 - loss: 5.1180
Epoch 9/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.1029 - loss: 4.8671
Epoch 10/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.1240 - loss: 4.6238
Epoch 11/500
485/485 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.1514 - loss: 4.3831 
Epoch 12/500
485/485

In [3]:
sonnet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 10, 100)             │         320,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 150)                 │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 3201)                │         483,351 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,862,155 (10.92 MB)

 Trainable params: 954,051 (3.64 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,908,104 (7.28 MB)